# 05. Conciliación global, control de reglas y reproducibilidad — Avance 03

# Proyecto A&C — Avance 03
**Análisis de accidentes y hechos de tránsito en la Ciudad de México**  
**Equipo:** Méndez Cabral Ángel David y Ruiz Moreno Citlalli Viviana · UACM · Inteligencia de Negocios · Grupo 301 · 2026-2

## Objetivo
Consolidar las conciliaciones y bitácoras disponibles de SSC, C5 e INEGI-ATUS. Auditar que los archivos exportados puedan leerse y comprobar que pendientes y excluidos conserven su motivo y regla.

**Criterio fundamental:** duplicados exactos retirados, excluidos por regla y pendientes son categorías distintas. Los folios repetidos son una marca de revisión y no se suman como registros adicionales. No se suman registros de fuentes distintas.

Este notebook no inventa resultados ni considera una fuente terminada si todavía no tiene una conciliación válida.

> **Estado de las salidas guardadas:** las salidas incrustadas en este notebook corresponden a otra ruta de ejecución y reportan que no hay conciliación SSC. No acreditan la auditoría de las corridas listadas en el informe y en los notebooks ETL. Como esta carpeta no incluye los datos procesados ni las bitácoras exportadas, se requiere restaurar esos artefactos y volver a ejecutar este notebook para validar el estado actual.

## 1. Entorno y funciones de lectura/escritura

Se excluyen carpetas ocultas y checkpoints al localizar ejecuciones. Se admiten CSV normales y CSV comprimidos (`.csv.gz`). Los reportes globales también se guardan comprimidos para reducir el espacio utilizado.

In [1]:
from pathlib import Path
import pandas as pd
import csv
import gzip
import re

def localizar_raiz_proyecto():
    """Encuentra Avance03 aunque Jupyter se inicie desde otra ubicación."""
    actual = Path.cwd().resolve()
    candidatos = [actual, *actual.parents]
    for carpeta in candidatos:
        if carpeta.name.lower() == "avance03":
            return carpeta
    for carpeta in candidatos:
        if (carpeta / "data" / "processed").exists() or (carpeta / "data" / "raw").exists():
            return carpeta
    return actual

ROOT = localizar_raiz_proyecto()

BASE = ROOT / "data" / "processed"
FUENTES = ["SSC", "C5", "INEGI_ATUS"]
SALIDA_GLOBAL = BASE / "auditoria_global"
SALIDA_GLOBAL.mkdir(parents=True, exist_ok=True)


def es_carpeta_ejecucion(path):
    if not path.is_dir() or path.name.startswith("."):
        return False
    return bool(re.match(r"^\d{8}_\d{6}(?:_\d+)?$", path.name))


def leer_tabla(path):
    return pd.read_csv(
        path,
        dtype="string",
        encoding="utf-8-sig",
        compression="infer",
        low_memory=False
    )


def contar_filas(path):
    """Cuenta registros CSV sin cargar el archivo completo."""
    if str(path).endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8-sig", newline="") as archivo:
            lector = csv.reader(archivo)
            next(lector, None)
            return sum(1 for _ in lector)
    with open(path, "r", encoding="utf-8-sig", newline="") as archivo:
        lector = csv.reader(archivo)
        next(lector, None)
        return sum(1 for _ in lector)


def escribir_reporte(df, path):
    path = Path(path)
    if path.suffix != ".gz":
        path = path.with_suffix(path.suffix + ".gz")
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with gzip.open(path, "wt", encoding="utf-8-sig", newline="") as archivo:
            df.to_csv(archivo, index=False, chunksize=10000)
        if contar_filas(path) != len(df):
            raise AssertionError(f"No coincide el conteo al guardar {path.name}.")
        return path
    except Exception:
        if path.exists():
            path.unlink()
        raise


def buscar_archivo(run, nombres):
    for nombre in nombres:
        candidato = run / nombre
        if candidato.exists():
            return candidato
    return None

print("Directorio del proyecto:", ROOT)
print("Carpeta de datos procesados:", BASE)

Directorio del proyecto: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/Notebooks3/Avance03
Carpeta de datos procesados: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/Notebooks3/Avance03/data/processed


## 2. Identificar la ejecución más reciente por fuente

Solo se consideran carpetas con formato de ejecución `AAAAMMDD_HHMMSS` (o sufijo numérico) y se ignoran `.ipynb_checkpoints` y otras carpetas ocultas.

In [2]:
ejecuciones = {}
filas_estado = []

for fuente in FUENTES:
    carpeta = BASE / fuente
    runs = [p for p in carpeta.iterdir() if es_carpeta_ejecucion(p)] if carpeta.exists() else []
    runs = sorted(runs, key=lambda p: p.name)
    ultima = runs[-1] if runs else None
    ejecuciones[fuente] = ultima

    filas_estado.append({
        "fuente": fuente,
        "ejecucion": str(ultima) if ultima else "",
        "estado": "ejecución encontrada" if ultima else "pendiente de ejecutar ETL",
        "numero_ejecuciones_detectadas": len(runs)
    })

estado = pd.DataFrame(filas_estado)
display(estado)

,fuente,ejecucion,estado,numero_ejecuciones_detectadas
0,SSC,/home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Ava...,ejecución encontrada,1
1,C5,/home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Ava...,ejecución encontrada,2
2,INEGI_ATUS,/home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Ava...,ejecución encontrada,2


## 3. Consolidar conciliaciones

Se incorporan únicamente archivos `conciliacion_por_archivo.csv` o su versión comprimida. Una fuente sin ese archivo queda reportada como pendiente; no se generan cifras ficticias.

In [3]:
concils = []
fuentes_sin_conciliacion = []

for fuente, run in ejecuciones.items():
    if run is None:
        fuentes_sin_conciliacion.append(fuente)
        continue

    archivo = buscar_archivo(run, [
        "conciliacion_por_archivo.csv.gz",
        "conciliacion_por_archivo.csv"
    ])

    if archivo is None:
        fuentes_sin_conciliacion.append(fuente)
        continue

    df = leer_tabla(archivo)
    df.insert(0, "fuente", fuente)
    df.insert(1, "archivo_conciliacion", archivo.name)
    concils.append(df)

if concils:
    global_df = pd.concat(concils, ignore_index=True, sort=False)
    display(global_df)
    escribir_reporte(global_df, SALIDA_GLOBAL / "conciliacion_global.csv.gz")
else:
    global_df = pd.DataFrame()
    print("Aún no existen conciliaciones para consolidar.")

if fuentes_sin_conciliacion:
    print("Fuentes sin conciliación disponible:", ", ".join(fuentes_sin_conciliacion))

if not global_df.empty and "conciliacion_correcta" in global_df.columns:
    valores = global_df["conciliacion_correcta"].fillna("").str.lower().str.strip()
    global_df["conciliacion_correcta"] = valores.isin(["true", "1", "si", "sí"])
    display(global_df[["fuente", "conciliacion_correcta"]])

,fuente,archivo_conciliacion,archivo,registros_iniciales,duplicados_exactos_retirados,conservados,pendientes_primarios,excluidos,folios_repetidos_revision_superpuesta,sin_hora_revision_temporal_superpuesta,total_reconciliado,conciliacion_correcta,pendientes,filas_con_folio_repetido_revision
0,C5,conciliacion_por_archivo.csv.gz,incidentes_viales_C5_etapa3.csv,1341419,0,1341419,0,0,20,0,1341419,True,<NA>,<NA>
1,INEGI_ATUS,conciliacion_por_archivo.csv.gz,INEGI_ATUS_etapa3_conservados.csv,62424,0,62424,<NA>,0,<NA>,<NA>,62424,True,0,0


Fuentes sin conciliación disponible: SSC


,fuente,conciliacion_correcta
0,C5,True
1,INEGI_ATUS,True


## 4. Consolidar bitácoras

Busco bitácoras dentro de la ejecución procesada y, si no están allí, en la carpeta paralela `bitacora/<fuente>/<RUN_ID>`. Se admiten los nombres `bitacora.csv`, `bitacora_<fuente>.csv` y sus versiones `.gz`.

In [4]:
logs = []

for fuente, run in ejecuciones.items():
    if run is None:
        continue

    candidatos = [
        run / "bitacora.csv.gz",
        run / "bitacora.csv",
        run / f"bitacora_{fuente}.csv.gz",
        run / f"bitacora_{fuente}.csv"
    ]

    log_dir = ROOT / "bitacora" / fuente / run.name
    candidatos.extend([
        log_dir / "bitacora.csv.gz",
        log_dir / "bitacora.csv",
        log_dir / f"bitacora_{fuente}.csv.gz",
        log_dir / f"bitacora_{fuente}.csv"
    ])

    archivo_log = next((p for p in candidatos if p.exists()), None)
    if archivo_log is not None:
        df = leer_tabla(archivo_log)
        df.insert(0, "fuente_consolidada", fuente)
        logs.append(df)

if logs:
    bitacora_global = pd.concat(logs, ignore_index=True, sort=False)
    display(bitacora_global.head(20))
    escribir_reporte(bitacora_global, SALIDA_GLOBAL / "bitacora_global.csv.gz")
    print("Registros de bitácora consolidados:", len(bitacora_global))
else:
    bitacora_global = pd.DataFrame()
    print("No se encontraron bitácoras para consolidar.")

,fuente_consolidada,fecha_hora_local,fuente,archivo_origen,proceso,regla,registros_antes,registros_despues,decision,justificacion,detalle,version_proceso
0,C5,2026-11-09T02:28:04-06:00,C5,incidentes_viales_C5_etapa3.csv,Extracción,RG-01/RG-11,1341419,1341419,Conservar original y crear copia de trabajo,El archivo original no se modifica; se registr...,<NA>,Avance03-v3
1,C5,2026-11-09T02:28:18-06:00,C5,incidentes_viales_C5_etapa3.csv,Duplicados exactos,C5-R01/RG-04,1341419,1341419,Retirar copias exactas adicionales,Se conserva la primera aparición y se exporta ...,<NA>,Avance03-v3
2,C5,2026-11-09T02:28:18-06:00,C5,incidentes_viales_C5_etapa3.csv,Folios repetidos,C5-R02/RG-05,1341419,1341419,Conservar y marcar para revisión,La repetición del folio no basta para eliminar...,Filas incluidas en revisión: 20,Avance03-v3
3,C5,2026-11-09T02:29:54-06:00,C5,incidentes_viales_C5_etapa3.csv,Revisión temporal,C5-R04/C5-R05,1341419,1341419,Conservar sin hora para otros análisis,La falta de hora solo limita el análisis tempo...,Filas pendientes para franjas horarias: 0,Avance03-v3
4,C5,2026-11-09T02:30:58-06:00,C5,incidentes_viales_C5_etapa3.csv,Conciliación,RG-07/RG-11/RG-12,1341419,1341419,Conciliación correcta,Las categorías primarias son excluyentes y sum...,<NA>,Avance03-v3
5,INEGI_ATUS,2026-11-09T02:30:42-06:00,INEGI_ATUS,INEGI_ATUS_etapa3_conservados.csv,Extracción,RG-01/RG-11,62424,62424,Conservar copia de trabajo,El original no se modifica; se registra proced...,<NA>,Avance03-v3
6,INEGI_ATUS,2026-11-09T02:30:45-06:00,INEGI_ATUS,INEGI_ATUS_etapa3_conservados.csv,Duplicados exactos,RG-07,62424,62424,Retirar duplicados exactos,Se conserva la primera aparición; folios repet...,Duplicados exactos retirados: 0,Avance03-v3
7,INEGI_ATUS,2026-11-09T02:30:48-06:00,INEGI_ATUS,INEGI_ATUS_etapa3_conservados.csv,Validación de fechas,INEGI-R02,62424,62424,Clasificar registros,Solo se marcan fechas no vacías e ininterpreta...,Columna: No identificada; fechas inválidas: 0,Avance03-v3
8,INEGI_ATUS,2026-11-09T02:30:53-06:00,INEGI_ATUS,INEGI_ATUS_etapa3_conservados.csv,Conciliación,RG-07/RG-11/RG-12,62424,62424,Validar destino de cada fila,"Conservados, pendientes, excluidos y duplicado...",<NA>,Avance03-v3


Registros de bitácora consolidados: 9


## 5. Auditoría de archivos independientes

Reviso todos los CSV y CSV comprimidos de cada ejecución. La auditoría cuenta registros de forma secuencial y comprueba que el archivo pueda descomprimirse/interpretarse completamente.

In [5]:
auditoria = []

for fuente, run in ejecuciones.items():
    if run is None:
        continue

    archivos = sorted(list(run.glob("*.csv")) + list(run.glob("*.csv.gz")))
    for archivo in archivos:
        try:
            filas = contar_filas(archivo)
            columnas_archivo = pd.read_csv(
                archivo, dtype="string", encoding="utf-8-sig",
                compression="infer", nrows=0
            ).columns
            auditoria.append({
                "fuente": fuente,
                "archivo": archivo.name,
                "lectura": "correcta",
                "filas": filas,
                "columnas": len(columnas_archivo),
                "detalle": ""
            })
        except Exception as error:
            auditoria.append({
                "fuente": fuente,
                "archivo": archivo.name,
                "lectura": "error",
                "filas": None,
                "columnas": None,
                "detalle": str(error)
            })

auditoria_df = pd.DataFrame(auditoria)
display(auditoria_df)
if not auditoria_df.empty:
    escribir_reporte(auditoria_df, SALIDA_GLOBAL / "auditoria_archivos.csv.gz")

,fuente,archivo,lectura,filas,columnas,detalle
0,SSC,inventario_fuentes.csv,correcta,5,6,
1,C5,bitacora_C5.csv.gz,correcta,5,11,
2,C5,conciliacion_por_archivo.csv.gz,correcta,1,10,
3,C5,incidentes_viales_C5_etapa3_discrepancias_terr...,correcta,0,40,
4,C5,incidentes_viales_C5_etapa3_duplicados_exactos...,correcta,0,34,
5,C5,incidentes_viales_C5_etapa3_excluidos_por_regl...,correcta,0,40,
6,C5,incidentes_viales_C5_etapa3_folios_repetidos_r...,correcta,20,36,
7,C5,incidentes_viales_C5_etapa3_limpio.csv.gz,correcta,1341419,40,
8,C5,incidentes_viales_C5_etapa3_pendientes.csv.gz,correcta,0,40,
9,C5,incidentes_viales_C5_etapa3_sin_hora_ocurrenci...,correcta,0,40,


## 6. Verificar justificación de pendientes y excluidos

Los archivos de pendientes y excluidos deben incluir `_motivo_decision` y `_regla_aplicada`. Los duplicados exactos y los folios repetidos se auditan como categorías distintas.

In [6]:
revisiones = []

for fuente, run in ejecuciones.items():
    if run is None:
        continue

    patrones = ["*_pendientes.csv", "*_pendientes.csv.gz",
                "*_excluidos_por_regla.csv", "*_excluidos_por_regla.csv.gz"]
    archivos_revision = []
    for patron in patrones:
        archivos_revision.extend(run.glob(patron))

    for archivo in sorted(set(archivos_revision)):
        try:
            columnas = list(pd.read_csv(
                archivo, dtype="string", encoding="utf-8-sig",
                compression="infer", nrows=0
            ).columns)
            faltan = [c for c in ["_motivo_decision", "_regla_aplicada"] if c not in columnas]
            filas = contar_filas(archivo)
            revisiones.append({
                "fuente": fuente,
                "archivo": archivo.name,
                "filas": filas,
                "campos_justificacion_faltantes": ";".join(faltan),
                "justificacion_completa": len(faltan) == 0
            })
        except Exception as error:
            revisiones.append({
                "fuente": fuente,
                "archivo": archivo.name,
                "filas": None,
                "campos_justificacion_faltantes": "No se pudo leer",
                "justificacion_completa": False
            })

revision_df = pd.DataFrame(revisiones)
display(revision_df)
if not revision_df.empty:
    escribir_reporte(revision_df, SALIDA_GLOBAL / "auditoria_justificaciones.csv.gz")

,fuente,archivo,filas,campos_justificacion_faltantes,justificacion_completa
0,C5,incidentes_viales_C5_etapa3_excluidos_por_regl...,0,,True
1,C5,incidentes_viales_C5_etapa3_pendientes.csv.gz,0,,True
2,INEGI_ATUS,INEGI_ATUS_etapa3_conservados_excluidos_por_re...,0,,True
3,INEGI_ATUS,INEGI_ATUS_etapa3_conservados_pendientes.csv.gz,0,,True


## 7. Resultado de auditoría y cierre

El notebook informa por separado qué fuentes tienen ejecución, conciliación, bitácora y archivos legibles. La ausencia de una fuente o conciliación no se oculta ni se sustituye con resultados inventados.

In [7]:
resumen_final = estado.copy()
resumen_final["conciliacion_disponible"] = resumen_final["fuente"].map(
    lambda f: bool(not global_df.empty and global_df["fuente"].eq(f).any()) if not global_df.empty else False
)
resumen_final["bitacora_disponible"] = resumen_final["fuente"].map(
    lambda f: bool(not bitacora_global.empty and bitacora_global["fuente_consolidada"].eq(f).any()) if not bitacora_global.empty else False
)
resumen_final["archivos_auditados"] = resumen_final["fuente"].map(
    lambda f: int(auditoria_df["fuente"].eq(f).sum()) if not auditoria_df.empty else 0
)

display(resumen_final)
escribir_reporte(resumen_final, SALIDA_GLOBAL / "estado_final_fuentes.csv.gz")

if not auditoria_df.empty and auditoria_df["lectura"].eq("error").any():
    print("ADVERTENCIA: hay archivos que no pudieron leerse; revisa auditoria_archivos.csv.gz.")
if not revision_df.empty and not revision_df["justificacion_completa"].all():
    print("ADVERTENCIA: hay archivos pendientes/excluidos sin los campos de justificación esperados.")

print("\nAuditoría global terminada.")
print("Reportes:", SALIDA_GLOBAL)
print("Fuentes sin conciliación:", fuentes_sin_conciliacion)

,fuente,ejecucion,estado,numero_ejecuciones_detectadas,conciliacion_disponible,bitacora_disponible,archivos_auditados
0,SSC,/home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Ava...,ejecución encontrada,1,False,False,1
1,C5,/home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Ava...,ejecución encontrada,2,True,True,11
2,INEGI_ATUS,/home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Ava...,ejecución encontrada,2,True,True,10



Auditoría global terminada.
Reportes: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/Notebooks3/Avance03/data/processed/auditoria_global
Fuentes sin conciliación: ['SSC']


## 8. Interpretación y limitaciones

La conciliación global confirma destinos por archivo y fuente, pero no autoriza sumar SSC, C5 e INEGI-ATUS. La carga a PostgreSQL y Power BI corresponde a fases posteriores. Una fuente solo se considera conciliada cuando su propio ETL generó una conciliación verificable.